# 03 — Interpreting Results

Deep-dive into the statistical outputs:
**SPA p-value**, **PSR (Probabilistic Sharpe Ratio)**, **FDR (False Discovery Rate)**, and **risk allocation weights**.

## SPA p-value (Superior Predictive Ability)

Hansen (2005) SPA tests whether the observed strategy edge is real or just the best of many random alternatives.

**Interpretation tiers (per-symbol SPA context):**
- `p < 0.0025` → PROD (very strong evidence)
- `0.0025 <= p < 0.025` → TRADE (strong evidence)
- `0.025 <= p < 0.075` → WATCH (moderate evidence)
- `0.075 <= p < 0.15` → RESEARCH (weak evidence)
- `p >= 0.15` → NO EDGE

**Caveats:**
- SPA accounts for cross-symbol multiple comparisons.
- Low p-value + small sample size = noise. Always inspect trade count (need >=30 for reliable inference).
- SPA does NOT replace live trading validation.

## PSR (Probabilistic Sharpe Ratio)

In [ ]:
from quant_lib.core._testing import prob_sharpe_ratio, label_p_value
import numpy as np

rng = np.random.default_rng(42)
trade_pnl = rng.normal(0.5, 1.0, 100)  # synthetic 100 trades
sr, psr = prob_sharpe_ratio(trade_pnl, annualize=False)
print(f"Sharpe Ratio: {sr:.4f}")
print(f"PSR: {psr:.4f}")
label, conf, interp = label_p_value(1 - psr, context="mean_r")
print(f"Label: {label} ({conf})")

## FDR (Benjamini-Hochberg)

When testing MULTIPLE symbols, use FDR to control the false discovery rate (expected proportion of false positives).

In [ ]:
from quant_lib.core._testing import fdr_correction
import numpy as np

p_values = np.array([0.001, 0.05, 0.2, 0.4, 0.6])
rejected, q_values = fdr_correction(p_values, alpha=0.05)
print("Rejected:", rejected)
print("Adjusted p-values:", q_values)

# Try a higher alpha for more rejections
rejected_loose, q_values_loose = fdr_correction(p_values, alpha=0.15)
print(f"At alpha=0.15: rejected {rejected_loose.sum()}/{len(p_values)}")

## Risk allocation weights

Per-fold PF (profit factor) weighted risk allocation:
- Each symbol's position size scales with its recent PF
- Clamped to `[0.5, 1.5]` to prevent runaway concentration
- Total risk is preserved (renormalized across symbols)

Inspect with `result.risk_weights` (per-fold) or `extract_final_fold_weights(result.risk_weights)` for the live-trading weights.

## Next steps

- Review `docs/methodology.md` for full statistical theory.
- Use `print_candidate_report(result)` for a formatted summary.
- Use `plot_equity_curve(result.daily_equity)` to visualize.